In [1]:
import os
import json
from sklearn.manifold import MDS
import matplotlib.pyplot as plt
import torch
import numpy as np
from scipy.stats import spearmanr

import compcats.utils.responsecomparison_utils as rcutils

import main_analysis as ma
import paths

# Matplotlib settings
import matplotlib as mpl
from matplotlib import font_manager as fm
from matplotlib import rcParams
import matplotlib.colors as mcolors
rcParams.update({'figure.autolayout': False})
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
# Font
font_path = '/home/alishd/.fonts/Avenir.ttc'
avenir_font = fm.FontProperties(fname=font_path)
mpl.rcParams['font.family'] = avenir_font.get_name()
ttc = fm.get_font(font_path)
print("Font name from TTC:", ttc.family_name)
print("Style:", ttc.style_name)

RESULTS_DIR = 'results'
MODELS_DIR = 'models'
PLOTS_DIR = f'{RESULTS_DIR}/plots/postvss25'
os.makedirs(PLOTS_DIR, exist_ok=True)

NSD_DIR = paths.NSD_DIR
GRILLSPECTOR_LOCALIZER_DIR = paths.GRILLSPECTOR_LOCALIZER_DIR
KONKLE_LOCALIZER_DIR = paths.KONKLE_LOCALIZER_DIR
KANWISHER_LOCALIZER_DIR = paths.KANWISHER_LOCALIZER_DIR

model_fmri_frois = [
    ('FFA', 'FFA'),
    ('PPA', 'PPA'),
    ('EBA', 'EBA'),
]
fmri_frois = list({froi for froi, _ in model_fmri_frois})
tthreshold = 7
model_localizer = 'grillspector'

Font name from TTC: Avenir
Style: Book


In [2]:
# Util functions
def calculate_stress(original_distances, projected_distances):
    """
    Calculate the Kruskal's Stress between two distance matrices.
    
    Args:
    original_distances (torch.Tensor): The original distance matrix (n x n).
    projected_distances (torch.Tensor): The distance matrix after MDS (n x n).
    
    Returns:
    torch.Tensor: The stress value.
    """
    # Compute the squared differences between the original and projected distances
    diff = original_distances - projected_distances
    squared_diff = diff ** 2
    
    # Calculate the sum of squared differences
    numerator = torch.sum(squared_diff)
    
    # Calculate the sum of squared original distances
    denominator = torch.sum(original_distances ** 2)
    
    # Calculate stress
    stress = torch.sqrt(numerator / denominator)
    
    return stress.item()

In [5]:
# Get data
subjects_data, subjects_splithalf_pearsonrs, subjects_splithalf_rdmrs = \
    ma.get_allsubjects_nsd515_data(fmri_frois, tthreshold)
# for froi in subjects_data.keys():
#     for subject in subjects_data[froi].keys():
#         subjects_data[froi][subject] = subjects_data[froi][subject].mean(axis=-1).cuda()
models_data = ma.get_allmodels_nsd515_data(
    model_fmri_frois, tthreshold, model_localizer, unitmean=False
)

Using cache found in /home/alishd/.cache/torch/hub/harvard-visionlab_open_ipcl_main


/home/alishd/.cache/torch/checkpoints/1a6IO4xhWy8cdyAuWv0BxSONz7lMcKbZE.pth.tar


/om2/user/alishd/anaconda/envs/catsel/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/om2/user/alishd/anaconda/envs/catsel/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Using cached ./models


No pretrained configuration specified for vit_base_patch16_224_in21k model. Using a default. Please add a config to the model pretrained_cfg registry or pass explicitly.
Using cache found in /home/alishd/.cache/torch/hub/facebookresearch_dinov2_main
/home/alishd/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/home/alishd/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/home/alishd/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")
Using cache found in /home/alishd/.cache/torch/hub/facebookresearch_dinov2_main
Using cache found in /home/alishd/.cache/torch/hub/harvard-visionlab_open_ipcl_main


...

Using cache found in /home/alishd/.cache/torch/hub/facebookresearch_dinov2_main
Using cache found in /home/alishd/.cache/torch/hub/facebookresearch_dinov2_main
Using cache found in /home/alishd/.cache/torch/hub/harvard-visionlab_open_ipcl_main


/home/alishd/.cache/torch/checkpoints/1a6IO4xhWy8cdyAuWv0BxSONz7lMcKbZE.pth.tar


/om2/user/alishd/anaconda/envs/catsel/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/om2/user/alishd/anaconda/envs/catsel/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Using cached ./models


Using cache found in /home/alishd/.cache/torch/hub/facebookresearch_dinov2_main
Using cache found in /home/alishd/.cache/torch/hub/facebookresearch_dinov2_main


# Noise-corrected

In [6]:
# Make folders for saving results
results_dir = f'{RESULTS_DIR}/postvss25_mds/noisecorrected'
plots_dir = f'{PLOTS_DIR}/fig3/mds/noisecorrected'
os.makedirs(results_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)

In [5]:
# Calculate distances for each froi
froi_pearsronr_noisecorrecteddistances = None
froi_rdmr_noisecorrecteddistances = None
for model_froi, fmri_froi in model_fmri_frois:
    print(f'Computing distances for {model_froi} and {fmri_froi}')
    
    # MDS for pearson correlation distances
    print('Computing MDS for Pearson correlation distances...')
    # Subject data for the specific froi
    froi_subjects_data = subjects_data[fmri_froi]
    # Take mean across voxels for each subject
    froi_subjects_data = {k: v.mean(axis=-1).cuda() for k, v in froi_subjects_data.items()}
    # Model data for the specific froi
    froi_models_data = models_data[(model_froi, fmri_froi)]
    # Take mean across units for each model
    froi_models_data = {k: v.mean(axis=-1).cuda() for k, v in froi_models_data.items()}
    # Get split-half Pearson correlation values for the froi
    froi_subjects_splithalf_pearsonrs = subjects_splithalf_pearsonrs[fmri_froi]

    keys = list(froi_subjects_data.keys()) + list(froi_models_data.keys())
    froi_splithalf_pearsonr_vector = []
    for key in keys:
        if key in froi_subjects_splithalf_pearsonrs:
            froi_splithalf_pearsonr_vector.append(froi_subjects_splithalf_pearsonrs[key])
        elif key in froi_models_data:
            froi_splithalf_pearsonr_vector.append(1)

    # Compute reliability matrix
    reliability_vector = torch.tensor(froi_splithalf_pearsonr_vector)
    reliability_matrix = torch.sqrt(reliability_vector[:, None] * reliability_vector[None, :])
    # Ensure diagonal elements are 1
    diag_indices = torch.arange(reliability_matrix.shape[0])
    reliability_matrix[diag_indices, diag_indices] = 1.0

    responses = []
    for key in keys:
        if key in froi_subjects_data:
            responses.append(froi_subjects_data[key])
        elif key in froi_models_data:
            responses.append(froi_models_data[key])
    responses = torch.stack(responses, dim=0)

    # Compute pairwise distances
    n_systems = responses.shape[0]
    distances = torch.zeros((n_systems, n_systems))
    for i in range(n_systems):
        for j in range(i + 1, n_systems):  # Compute only upper triangular (symmetric)
            dist = 1 - (rcutils.pearsonr(responses[i], responses[j]) / reliability_matrix[i, j])
            distances[i, j] = dist
            distances[j, i] = dist  # Symmetric matrix
    
    # Save the distances
    mds_results = {
        'model_froi': model_froi,
        'fmri_froi': fmri_froi,
        'tthreshold': tthreshold,
        'model_localizer': model_localizer,
        'dataset': 'nsd515',
        'keys': keys,
    }
    distances_file = f'{results_dir}/noisecorrected_distances_pearsonr_{model_froi}_{fmri_froi}.pt'
    torch.save(distances.cpu(), distances_file)
    # Save the results dictionary as json
    with open(f'{results_dir}/noisecorrected_distances_pearsonr_{model_froi}_{fmri_froi}_info.json', 'w') as f:
        json.dump(mds_results, f)
    # Save the distances for later use
    if froi_pearsronr_noisecorrecteddistances is None:
        froi_pearsronr_noisecorrecteddistances = distances.unsqueeze(0)
    else:
        froi_pearsronr_noisecorrecteddistances = torch.cat(
            (froi_pearsronr_noisecorrecteddistances, distances.unsqueeze(0)), dim=0
        )
    
    # MDS for pearson RDM distances
    print('Computing distances for RDM correlation distances...')
    # Subject data for the specific froi
    froi_subjects_data = subjects_data[fmri_froi]
    # Take mean across voxels for each subject
    froi_subjects_data = {k: v.cuda() for k, v in froi_subjects_data.items()}
    # Model data for the specific froi
    froi_models_data = models_data[(model_froi, fmri_froi)]
    # Take mean across units for each model
    froi_models_data = {k: v.cuda() for k, v in froi_models_data.items()}
    # Get split-half Pearson correlation values for the froi
    froi_subjects_splithalf_rdmrs = subjects_splithalf_rdmrs[fmri_froi]

    keys = list(froi_subjects_data.keys()) + list(froi_models_data.keys())
    froi_splithalf_rdmr_vector = []
    for key in keys:
        if key in froi_subjects_splithalf_rdmrs:
            froi_splithalf_rdmr_vector.append(froi_subjects_splithalf_rdmrs[key])
        elif key in froi_models_data:
            froi_splithalf_rdmr_vector.append(1)

    # Compute reliability matrix
    reliability_vector = torch.tensor(froi_splithalf_rdmr_vector)
    reliability_matrix = torch.sqrt(reliability_vector[:, None] * reliability_vector[None, :])
    # Ensure diagonal elements are 1
    diag_indices = torch.arange(reliability_matrix.shape[0])
    reliability_matrix[diag_indices, diag_indices] = 1.0

    responses = []
    for key in keys:
        if key in froi_subjects_data:
            responses.append(froi_subjects_data[key])
        elif key in froi_models_data:
            responses.append(froi_models_data[key])
    # responses = torch.stack(responses, dim=0)

    # Compute pairwise distances
    n_systems = len(keys)
    distances = torch.zeros((n_systems, n_systems))
    for i in range(n_systems):
        for j in range(i + 1, n_systems):  # Compute only upper triangular (symmetric)
            dist = 1 - (rcutils.rdmr(responses[i], responses[j]) / reliability_matrix[i, j])
            distances[i, j] = dist
            distances[j, i] = dist  # Symmetric matrix
    
    # Save the distances
    mds_results = {
        'model_froi': model_froi,
        'fmri_froi': fmri_froi,
        'tthreshold': tthreshold,
        'model_localizer': model_localizer,
        'dataset': 'nsd515',
        'keys': keys,
    }
    distances_file = f'{results_dir}/noisecorrected_distances_rdmr_{model_froi}_{fmri_froi}.pt'
    torch.save(distances.cpu(), distances_file)
    # Save the results dictionary as json
    with open(f'{results_dir}/noisecorrected_results_rdmr_{model_froi}_{fmri_froi}.json', 'w') as f:
        json.dump(mds_results, f)
    # Save the distances for later use
    if froi_rdmr_noisecorrecteddistances is None:
        froi_rdmr_noisecorrecteddistances = distances.unsqueeze(0)
    else:
        froi_rdmr_noisecorrecteddistances = torch.cat(
            (froi_rdmr_noisecorrecteddistances, distances.unsqueeze(0)), dim=0
        )

# Take the median of the distances across frois
froi_pearsronr_noisecorrecteddistances = froi_pearsronr_noisecorrecteddistances.median(dim=0).values
froi_rdmr_noisecorrecteddistances = froi_rdmr_noisecorrecteddistances.median(dim=0).values
# Save the median distances
torch.save(
    froi_pearsronr_noisecorrecteddistances.cpu(),
    f'{results_dir}/noisecorrected_distances_pearsonr_median_median.pt'
)
torch.save(
    froi_rdmr_noisecorrecteddistances.cpu(),
    f'{results_dir}/noisecorrected_distances_rdmr_median_median.pt'
)

Computing distances for FFA and FFA
Computing MDS for Pearson correlation distances...
Computing distances for RDM correlation distances...
Computing distances for PPA and PPA
Computing MDS for Pearson correlation distances...
Computing distances for RDM correlation distances...
Computing distances for EBA and EBA
Computing MDS for Pearson correlation distances...
Computing distances for RDM correlation distances...


In [6]:
# MDS
mds_frois = model_fmri_frois + [('median', 'median')]
for model_froi, fmri_froi in mds_frois:
    print(f'Computing MDS for {model_froi} and {fmri_froi}')
    distances_dict = {
        'pearsonr': torch.load(f'{results_dir}/noisecorrected_distances_pearsonr_{model_froi}_{fmri_froi}.pt'),
        'rdmr': torch.load(f'{results_dir}/noisecorrected_distances_rdmr_{model_froi}_{fmri_froi}.pt')
    }
    if model_froi == 'median' and fmri_froi == 'median':
        distances_info_dict = {
            'pearsonr': json.load(open(f'{results_dir}/noisecorrected_distances_pearsonr_FFA_FFA_info.json')),
            'rdmr': json.load(open(f'{results_dir}/noisecorrected_results_rdmr_FFA_FFA.json'))
        }
    else:
        distances_info_dict = {
            'pearsonr': json.load(open(f'{results_dir}/noisecorrected_distances_pearsonr_{model_froi}_{fmri_froi}_info.json')),
            'rdmr': json.load(open(f'{results_dir}/noisecorrected_results_rdmr_{model_froi}_{fmri_froi}.json'))
        }
    for dist_type, distances in distances_dict.items():
        distances_info = distances_info_dict[dist_type]
        if not (model_froi == 'median' and fmri_froi == 'median'):
            assert distances_info['model_froi'] == model_froi
            assert distances_info['fmri_froi'] == fmri_froi
        assert distances_info['tthreshold'] == tthreshold
        assert distances_info['model_localizer'] == model_localizer
        mds_results = {
            'model_froi': model_froi,
            'fmri_froi': fmri_froi,
            'tthreshold': tthreshold,
            'model_localizer': model_localizer,
            'dataset': 'nsd515',
            'keys': distances_info['keys'],
        }
        epsilon = 1e-5
        mds_results['epsilon'] = epsilon
        print(f'{distances[distances<=0].numel()} out of {distances.numel()} values are negative, clamping to {epsilon}')
        mds_results['total_negative_distances_clamped'] = distances[distances <= 0].numel()
        mds_results['total_distances'] = distances.numel()
        distances = torch.clamp(distances, min=epsilon)

        mds = MDS(n_components=2, dissimilarity='precomputed')
        mds_coords = mds.fit_transform(distances.cpu().numpy())
        mds_results['mds_coordinates'] = mds_coords.tolist()

        # Test if MDS distances are correct
        mds_coords_tensor = torch.tensor(mds_coords)
        n_systems = len(keys)
        lowdim_distances = torch.zeros((n_systems, n_systems))
        for i in range(n_systems):
            for j in range(i + 1, n_systems):  # Compute only upper triangular (symmetric)
                dist = torch.norm(mds_coords_tensor[i] - mds_coords_tensor[j])
                lowdim_distances[i, j] = dist
                lowdim_distances[j, i] = dist  # Symmetric matrix
        stress = calculate_stress(distances, lowdim_distances)
        print('Stress:', stress)
        mds_results['stress'] = stress

        # Find which indices correspond to subjects and which to models
        subject_indices = [i for i, key in enumerate(keys) if key in froi_subjects_data]
        model_indices = [i for i, key in enumerate(keys) if key in froi_models_data]

        # Calculate mean distances from each model to all subjects
        model_to_subjects_orig = torch.zeros(len(model_indices))
        model_to_subjects_mds = torch.zeros(len(model_indices))

        for i, model_idx in enumerate(model_indices):
            # Mean distance from this model to all subjects in original space
            model_to_subjects_orig[i] = distances[model_idx, subject_indices].mean()
            
            # Mean distance from this model to all subjects in MDS space
            model_to_subjects_mds[i] = lowdim_distances[model_idx, subject_indices].mean()

        # Calculate Spearman rank correlation
        sp_corr, p_value = spearmanr(model_to_subjects_orig.numpy(), model_to_subjects_mds.numpy())

        print(f"Spearman rank correlation between original and MDS distances: {sp_corr:.3f} (p={p_value:.3e})")
        mds_results['spearman_correlation_lowdim_orig_corr'] = sp_corr
        mds_results['spearman_correlation_lowdim_orig_p_value'] = p_value
        # Save the MDS results
        with open(f'{results_dir}/noisecorrected_mds_results_{dist_type}_{model_froi}_{fmri_froi}.json', 'w') as f:
            json.dump(mds_results, f)

Computing MDS for FFA and FFA
59 out of 1849 values are negative, clamping to 1e-05
Stress: 0.20032650232315063
Spearman rank correlation between original and MDS distances: 0.934 (p=2.500e-16)
75 out of 1849 values are negative, clamping to 1e-05
Stress: 0.26967671513557434
Spearman rank correlation between original and MDS distances: 0.827 (p=8.823e-10)
Computing MDS for PPA and PPA
81 out of 1849 values are negative, clamping to 1e-05
Stress: 0.2521415948867798
Spearman rank correlation between original and MDS distances: 0.803 (p=6.292e-09)
85 out of 1849 values are negative, clamping to 1e-05
Stress: 0.28769490122795105
Spearman rank correlation between original and MDS distances: 0.745 (p=2.793e-07)
Computing MDS for EBA and EBA
69 out of 1849 values are negative, clamping to 1e-05
Stress: 0.2557840943336487
Spearman rank correlation between original and MDS distances: 0.837 (p=3.631e-10)
91 out of 1849 values are negative, clamping to 1e-05
Stress: 0.2813813388347626
Spearman ra

In [ ]:
# Plot
for metric in ['pearsonr', 'rdmr']:
    # Median
    model_froi = fmri_froi = 'median'
    print(f'Plotting MDS for {model_froi} and {fmri_froi}, metric: {metric}')
    mds_results = json.load(
        open(f'{results_dir}/noisecorrected_mds_results_{metric}_{model_froi}_{fmri_froi}.json')
    )
    mds_coords = np.array(mds_results['mds_coordinates'])
    keys = mds_results['keys']

    human_color = 'royalblue'
    model_color = 'red'
    facecolors = [mcolors.to_rgba(human_color, alpha=0.75) if key in range(1, 9) else mcolors.to_rgba(model_color, alpha=0.75) for key in keys]
    plt.figure(figsize=(5, 5), dpi=300)
    plt.scatter(mds_coords[:, 0], mds_coords[:, 1], fc=facecolors, ec='none', s=100, linewidths=2)

    annotate_models = False  # Set to True to annotate models, False to not annotate
    if annotate_models:
        for i, key in enumerate(keys):
            if key in froi_models_data:
                plt.annotate(key, (mds_coords[i, 0], mds_coords[i, 1]), fontsize=8)

    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    plt.xticks([])
    plt.yticks([])
    # plt.xlabel('MDS Dimension 1')
    # plt.ylabel('MDS Dimension 2')
    # plt.legend(handles=[plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=human_color, markersize=10, label='Human'),
    #                     plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=model_color, markersize=10, label='Model')],
    #         loc='best')
    # plt.title(f'{model_froi} and {fmri_froi}')
    plt.axis('off')
    plt.gcf().patch.set_alpha(0.0)
    plt.tight_layout()
    # plt.show()
    plt.savefig(f'{plots_dir}/mds_noisecorrected_{metric}_median_frois.pdf', dpi=300, bbox_inches='tight', transparent=True)
    plt.close()

# Individual frois
fig, axs = plt.subplots(2, len(model_fmri_frois), figsize=(5*len(model_fmri_frois), 5*2), dpi=300)
for midx, metric in enumerate(['pearsonr', 'rdmr']):
    for fidx, (model_froi, fmri_froi) in enumerate(model_fmri_frois):
        print(f'Plotting MDS for {model_froi} and {fmri_froi}, metric: {metric}')
        mds_results = json.load(
            open(f'{results_dir}/noisecorrected_mds_results_{metric}_{model_froi}_{fmri_froi}.json')
        )
        mds_coords = np.array(mds_results['mds_coordinates'])
        keys = mds_results['keys']

        human_color = 'royalblue'
        model_color = 'red'
        # colors = [human_color if key in froi_subjects_data else model_color for key in keys]
        facecolors = [mcolors.to_rgba(human_color, alpha=0.75) if key in range(1, 9) else mcolors.to_rgba(model_color, alpha=0.75) for key in keys]
        # edgecolors = [human_color if key in froi_subjects_data else model_color for key in keys]
        # edgecolors = ['white' if key in froi_subjects_data else 'white' for key in keys]
        # plt.scatter(mds_coords[:, 0], mds_coords[:, 1], c=colors, s=100)
        axs[midx, fidx].scatter(mds_coords[:, 0], mds_coords[:, 1], fc=facecolors, ec='none', s=100, linewidths=2)

        annotate_models = False  # Set to True to annotate models, False to not annotate
        if annotate_models:
            for i, key in enumerate(keys):
                if key in froi_models_data:
                    plt.annotate(key, (mds_coords[i, 0], mds_coords[i, 1]), fontsize=8)

        # axs[midx, fidx].spines['top'].set_visible(False)
        # axs[midx, fidx].spines['right'].set_visible(False)
        axs[midx, fidx].set_xticks([])
        axs[midx, fidx].set_yticks([])
        # plt.xlabel('MDS Dimension 1')
        # plt.ylabel('MDS Dimension 2')
        # plt.legend(handles=[plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=human_color, markersize=10, label='Human'),
        #                     plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=model_color, markersize=10, label='Model')],
        #         loc='best')
        # axs[midx, fidx].set_title(f'{model_froi} and {fmri_froi}')
        axs[midx, fidx].axis('off')
        # Write froi name in bottom left corner
        axs[midx, fidx].text(0.9, 0.1, model_froi, transform=axs[midx, fidx].transAxes, fontsize=12, verticalalignment='top')
plt.gcf().patch.set_alpha(0.0)
plt.tight_layout()
# plt.show()
plt.savefig(f'{plots_dir}/mds_noisecorrected_pearsronr_rdmr_individual_frois.pdf', dpi=300, bbox_inches='tight', transparent=True)
plt.close()

Plotting MDS for FFA and FFA, metric: pearsonr
Plotting MDS for PPA and PPA, metric: pearsonr
Plotting MDS for EBA and EBA, metric: pearsonr
Plotting MDS for FFA and FFA, metric: rdmr
Plotting MDS for PPA and PPA, metric: rdmr
Plotting MDS for EBA and EBA, metric: rdmr


# Unocrrected

In [21]:
# Make folders for saving results
results_dir = f'{RESULTS_DIR}/postvss25_mds/uncorrected'
plots_dir = f'{PLOTS_DIR}/fig3/mds/uncorrected'
os.makedirs(results_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)

In [9]:
# Distances
froi_pearsronr_uncorrecteddistances = None
froi_rdmr_uncorrecteddistances = None
for model_froi, fmri_froi in model_fmri_frois:
    print(f'Computing distances for {model_froi} and {fmri_froi}')
    
    # MDS for pearson correlation distances
    print('Computing MDS for Pearson correlation distances...')
    # Subject data for the specific froi
    froi_subjects_data = subjects_data[fmri_froi]
    # Take mean across voxels for each subject
    froi_subjects_data = {k: v.mean(axis=-1).cuda() for k, v in froi_subjects_data.items()}
    # Model data for the specific froi
    froi_models_data = models_data[(model_froi, fmri_froi)]
    # Take mean across units for each model
    froi_models_data = {k: v.mean(axis=-1).cuda() for k, v in froi_models_data.items()}
    # Get split-half Pearson correlation values for the froi
    froi_subjects_splithalf_pearsonrs = subjects_splithalf_pearsonrs[fmri_froi]

    keys = list(froi_subjects_data.keys()) + list(froi_models_data.keys())
    froi_splithalf_pearsonr_vector = []
    for key in keys:
        if key in froi_subjects_splithalf_pearsonrs:
            froi_splithalf_pearsonr_vector.append(froi_subjects_splithalf_pearsonrs[key])
        elif key in froi_models_data:
            froi_splithalf_pearsonr_vector.append(1)

    # Compute reliability matrix
    reliability_vector = torch.tensor(froi_splithalf_pearsonr_vector)
    reliability_matrix = torch.sqrt(reliability_vector[:, None] * reliability_vector[None, :])
    # Ensure diagonal elements are 1
    diag_indices = torch.arange(reliability_matrix.shape[0])
    reliability_matrix[diag_indices, diag_indices] = 1.0

    responses = []
    for key in keys:
        if key in froi_subjects_data:
            responses.append(froi_subjects_data[key])
        elif key in froi_models_data:
            responses.append(froi_models_data[key])
    responses = torch.stack(responses, dim=0)

    # Compute pairwise distances
    n_systems = responses.shape[0]
    distances = torch.zeros((n_systems, n_systems))
    for i in range(n_systems):
        for j in range(i + 1, n_systems):  # Compute only upper triangular (symmetric)
            dist = 1 - (rcutils.pearsonr(responses[i], responses[j]))
            distances[i, j] = dist
            distances[j, i] = dist  # Symmetric matrix
    
    # Save the distances
    mds_results = {
        'model_froi': model_froi,
        'fmri_froi': fmri_froi,
        'tthreshold': tthreshold,
        'model_localizer': model_localizer,
        'dataset': 'nsd515',
        'keys': keys,
    }
    distances_file = f'{results_dir}/uncorrected_distances_pearsonr_{model_froi}_{fmri_froi}.pt'
    torch.save(distances.cpu(), distances_file)
    # Save the results dictionary as json
    with open(f'{results_dir}/uncorrected_distances_pearsonr_{model_froi}_{fmri_froi}_info.json', 'w') as f:
        json.dump(mds_results, f)
    # Save the distances for later use
    if froi_pearsronr_uncorrecteddistances is None:
        froi_pearsronr_uncorrecteddistances = distances.unsqueeze(0)
    else:
        froi_pearsronr_uncorrecteddistances = torch.cat(
            (froi_pearsronr_uncorrecteddistances, distances.unsqueeze(0)), dim=0
        )
    
    # MDS for pearson RDM distances
    print('Computing MDS for RDM correlation distances...')
    # Subject data for the specific froi
    froi_subjects_data = subjects_data[fmri_froi]
    # Take mean across voxels for each subject
    froi_subjects_data = {k: v.cuda() for k, v in froi_subjects_data.items()}
    # Model data for the specific froi
    froi_models_data = models_data[(model_froi, fmri_froi)]
    # Take mean across units for each model
    froi_models_data = {k: v.cuda() for k, v in froi_models_data.items()}
    # Get split-half Pearson correlation values for the froi
    froi_subjects_splithalf_rdmrs = subjects_splithalf_rdmrs[fmri_froi]

    keys = list(froi_subjects_data.keys()) + list(froi_models_data.keys())
    froi_splithalf_rdmr_vector = []
    for key in keys:
        if key in froi_subjects_splithalf_rdmrs:
            froi_splithalf_rdmr_vector.append(froi_subjects_splithalf_rdmrs[key])
        elif key in froi_models_data:
            froi_splithalf_rdmr_vector.append(1)

    # Compute reliability matrix
    reliability_vector = torch.tensor(froi_splithalf_rdmr_vector)
    reliability_matrix = torch.sqrt(reliability_vector[:, None] * reliability_vector[None, :])
    # Ensure diagonal elements are 1
    diag_indices = torch.arange(reliability_matrix.shape[0])
    reliability_matrix[diag_indices, diag_indices] = 1.0

    responses = []
    for key in keys:
        if key in froi_subjects_data:
            responses.append(froi_subjects_data[key])
        elif key in froi_models_data:
            responses.append(froi_models_data[key])
    # responses = torch.stack(responses, dim=0)

    # Compute pairwise distances
    n_systems = len(keys)
    distances = torch.zeros((n_systems, n_systems))
    for i in range(n_systems):
        for j in range(i + 1, n_systems):  # Compute only upper triangular (symmetric)
            dist = 1 - (rcutils.rdmr(responses[i], responses[j]))
            distances[i, j] = dist
            distances[j, i] = dist  # Symmetric matrix
    
    # Save the distances
    mds_results = {
        'model_froi': model_froi,
        'fmri_froi': fmri_froi,
        'tthreshold': tthreshold,
        'model_localizer': model_localizer,
        'dataset': 'nsd515',
        'keys': keys,
    }
    distances_file = f'{results_dir}/uncorrected_distances_rdmr_{model_froi}_{fmri_froi}.pt'
    torch.save(distances.cpu(), distances_file)
    # Save the results dictionary as json
    with open(f'{results_dir}/uncorrected_results_rdmr_{model_froi}_{fmri_froi}.json', 'w') as f:
        json.dump(mds_results, f)
    # Save the distances for later use
    if froi_rdmr_uncorrecteddistances is None:
        froi_rdmr_uncorrecteddistances = distances.unsqueeze(0)
    else:
        froi_rdmr_uncorrecteddistances = torch.cat(
            (froi_rdmr_uncorrecteddistances, distances.unsqueeze(0)), dim=0
        )

# Take the median of the distances across frois
froi_pearsronr_uncorrecteddistances = froi_pearsronr_uncorrecteddistances.median(dim=0).values
froi_rdmr_uncorrecteddistances = froi_rdmr_uncorrecteddistances.median(dim=0).values
# Save the median distances
torch.save(
    froi_pearsronr_uncorrecteddistances.cpu(),
    f'{results_dir}/uncorrected_distances_pearsonr_median_median.pt'
)
torch.save(
    froi_rdmr_uncorrecteddistances.cpu(),
    f'{results_dir}/uncorrected_distances_rdmr_median_median.pt'
)

Computing distances for FFA and FFA
Computing MDS for Pearson correlation distances...
Computing MDS for RDM correlation distances...
Computing distances for PPA and PPA
Computing MDS for Pearson correlation distances...
Computing MDS for RDM correlation distances...
Computing distances for EBA and EBA
Computing MDS for Pearson correlation distances...
Computing MDS for RDM correlation distances...


In [10]:
# MDS
mds_frois = model_fmri_frois + [('median', 'median')]
for model_froi, fmri_froi in mds_frois:
    print(f'Computing MDS for {model_froi} and {fmri_froi}')
    distances_dict = {
        'pearsonr': torch.load(f'{results_dir}/uncorrected_distances_pearsonr_{model_froi}_{fmri_froi}.pt'),
        'rdmr': torch.load(f'{results_dir}/uncorrected_distances_rdmr_{model_froi}_{fmri_froi}.pt')
    }
    if model_froi == 'median' and fmri_froi == 'median':
        distances_info_dict = {
            'pearsonr': json.load(open(f'{results_dir}/uncorrected_distances_pearsonr_FFA_FFA_info.json')),
            'rdmr': json.load(open(f'{results_dir}/uncorrected_results_rdmr_FFA_FFA.json'))
        }
    else:
        distances_info_dict = {
            'pearsonr': json.load(open(f'{results_dir}/uncorrected_distances_pearsonr_{model_froi}_{fmri_froi}_info.json')),
            'rdmr': json.load(open(f'{results_dir}/uncorrected_results_rdmr_{model_froi}_{fmri_froi}.json'))
        }
    for dist_type, distances in distances_dict.items():
        distances_info = distances_info_dict[dist_type]
        if not (model_froi == 'median' and fmri_froi == 'median'):
            assert distances_info['model_froi'] == model_froi
            assert distances_info['fmri_froi'] == fmri_froi
        assert distances_info['tthreshold'] == tthreshold
        assert distances_info['model_localizer'] == model_localizer
        mds_results = {
            'model_froi': model_froi,
            'fmri_froi': fmri_froi,
            'tthreshold': tthreshold,
            'model_localizer': model_localizer,
            'dataset': 'nsd515',
            'keys': distances_info['keys'],
        }
        epsilon = 1e-5
        mds_results['epsilon'] = epsilon
        print(f'{distances[distances<=0].numel()} out of {distances.numel()} values are negative, clamping to {epsilon}')
        mds_results['total_negative_distances_clamped'] = distances[distances <= 0].numel()
        mds_results['total_distances'] = distances.numel()
        distances = torch.clamp(distances, min=epsilon)

        mds = MDS(n_components=2, dissimilarity='precomputed')
        mds_coords = mds.fit_transform(distances.cpu().numpy())
        mds_results['mds_coordinates'] = mds_coords.tolist()

        # Test if MDS distances are correct
        mds_coords_tensor = torch.tensor(mds_coords)
        n_systems = len(keys)
        lowdim_distances = torch.zeros((n_systems, n_systems))
        for i in range(n_systems):
            for j in range(i + 1, n_systems):  # Compute only upper triangular (symmetric)
                dist = torch.norm(mds_coords_tensor[i] - mds_coords_tensor[j])
                lowdim_distances[i, j] = dist
                lowdim_distances[j, i] = dist  # Symmetric matrix
        stress = calculate_stress(distances, lowdim_distances)
        print('Stress:', stress)
        mds_results['stress'] = stress

        # Find which indices correspond to subjects and which to models
        subject_indices = [i for i, key in enumerate(keys) if key in froi_subjects_data]
        model_indices = [i for i, key in enumerate(keys) if key in froi_models_data]

        # Calculate mean distances from each model to all subjects
        model_to_subjects_orig = torch.zeros(len(model_indices))
        model_to_subjects_mds = torch.zeros(len(model_indices))

        for i, model_idx in enumerate(model_indices):
            # Mean distance from this model to all subjects in original space
            model_to_subjects_orig[i] = distances[model_idx, subject_indices].mean()
            
            # Mean distance from this model to all subjects in MDS space
            model_to_subjects_mds[i] = lowdim_distances[model_idx, subject_indices].mean()

        # Calculate Spearman rank correlation
        sp_corr, p_value = spearmanr(model_to_subjects_orig.numpy(), model_to_subjects_mds.numpy())

        print(f"Spearman rank correlation between original and MDS distances: {sp_corr:.3f} (p={p_value:.3e})")
        mds_results['spearman_correlation_lowdim_orig_corr'] = sp_corr
        mds_results['spearman_correlation_lowdim_orig_p_value'] = p_value
        # Save the MDS results
        with open(f'{results_dir}/uncorrected_mds_results_{dist_type}_{model_froi}_{fmri_froi}.json', 'w') as f:
            json.dump(mds_results, f)

Computing MDS for FFA and FFA
43 out of 1849 values are negative, clamping to 1e-05
Stress: 0.20142844319343567
Spearman rank correlation between original and MDS distances: 0.921 (p=4.850e-15)
43 out of 1849 values are negative, clamping to 1e-05
Stress: 0.28778448700904846
Spearman rank correlation between original and MDS distances: 0.737 (p=4.521e-07)
Computing MDS for PPA and PPA
43 out of 1849 values are negative, clamping to 1e-05
Stress: 0.24422533810138702
Spearman rank correlation between original and MDS distances: 0.822 (p=1.464e-09)
43 out of 1849 values are negative, clamping to 1e-05
Stress: 0.306560218334198
Spearman rank correlation between original and MDS distances: 0.468 (p=4.602e-03)
Computing MDS for EBA and EBA
43 out of 1849 values are negative, clamping to 1e-05
Stress: 0.2658149302005768
Spearman rank correlation between original and MDS distances: 0.628 (p=5.372e-05)
43 out of 1849 values are negative, clamping to 1e-05
Stress: 0.30771899223327637
Spearman ra

In [11]:
# Plot
for metric in ['pearsonr', 'rdmr']:
    # Median
    model_froi = fmri_froi = 'median'
    print(f'Plotting MDS for {model_froi} and {fmri_froi}, metric: {metric}')
    mds_results = json.load(
        open(f'{results_dir}/uncorrected_mds_results_{metric}_{model_froi}_{fmri_froi}.json')
    )
    mds_coords = np.array(mds_results['mds_coordinates'])
    keys = mds_results['keys']

    human_color = 'royalblue'
    model_color = 'red'
    # colors = [human_color if key in froi_subjects_data else model_color for key in keys]
    facecolors = [mcolors.to_rgba(human_color, alpha=0.75) if key in froi_subjects_data else mcolors.to_rgba(model_color, alpha=0.75) for key in keys]
    plt.figure(figsize=(5, 5), dpi=300)
    plt.scatter(mds_coords[:, 0], mds_coords[:, 1], fc=facecolors, ec='none', s=100, linewidths=2)

    annotate_models = False  # Set to True to annotate models, False to not annotate
    if annotate_models:
        for i, key in enumerate(keys):
            if key in froi_models_data:
                plt.annotate(key, (mds_coords[i, 0], mds_coords[i, 1]), fontsize=8)

    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    plt.xticks([])
    plt.yticks([])
    # plt.xlabel('MDS Dimension 1')
    # plt.ylabel('MDS Dimension 2')
    # plt.legend(handles=[plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=human_color, markersize=10, label='Human'),
    #                     plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=model_color, markersize=10, label='Model')],
    #         loc='best')
    # plt.title(f'{model_froi} and {fmri_froi}')
    plt.axis('off')
    plt.gcf().patch.set_alpha(0.0)
    plt.tight_layout()
    # plt.show()
    plt.savefig(f'{plots_dir}/mds_uncorrected_{metric}_median_frois.pdf', dpi=300, bbox_inches='tight', transparent=True)
    plt.close()

    # Individual frois
    fig, axs = plt.subplots(1, len(model_fmri_frois), figsize=(5*len(model_fmri_frois), 5), dpi=300)
    for fidx, (model_froi, fmri_froi) in enumerate(model_fmri_frois):
        print(f'Plotting MDS for {model_froi} and {fmri_froi}, metric: {metric}')
        mds_results = json.load(
            open(f'{results_dir}/uncorrected_mds_results_{metric}_{model_froi}_{fmri_froi}.json')
        )
        mds_coords = np.array(mds_results['mds_coordinates'])
        keys = mds_results['keys']

        human_color = 'royalblue'
        model_color = 'red'
        # colors = [human_color if key in froi_subjects_data else model_color for key in keys]
        facecolors = [mcolors.to_rgba(human_color, alpha=0.75) if key in froi_subjects_data else mcolors.to_rgba(model_color, alpha=0.75) for key in keys]
        # edgecolors = [human_color if key in froi_subjects_data else model_color for key in keys]
        edgecolors = ['white' if key in froi_subjects_data else 'white' for key in keys]
        # plt.scatter(mds_coords[:, 0], mds_coords[:, 1], c=colors, s=100)
        axs[fidx].scatter(mds_coords[:, 0], mds_coords[:, 1], fc=facecolors, ec='none', s=100, linewidths=2)

        annotate_models = False  # Set to True to annotate models, False to not annotate
        if annotate_models:
            for i, key in enumerate(keys):
                if key in froi_models_data:
                    plt.annotate(key, (mds_coords[i, 0], mds_coords[i, 1]), fontsize=8)

        axs[fidx].spines['top'].set_visible(False)
        axs[fidx].spines['right'].set_visible(False)
        axs[fidx].set_xticks([])
        axs[fidx].set_yticks([])
        # plt.xlabel('MDS Dimension 1')
        # plt.ylabel('MDS Dimension 2')
        # plt.legend(handles=[plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=human_color, markersize=10, label='Human'),
        #                     plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=model_color, markersize=10, label='Model')],
        #         loc='best')
        axs[fidx].set_title(f'{model_froi} and {fmri_froi}')
        axs[fidx].axis('off')
    plt.gcf().patch.set_alpha(0.0)
    plt.tight_layout()
    # plt.show()
    plt.savefig(f'{plots_dir}/mds_uncorrected_{metric}_individual_frois.pdf', dpi=300, bbox_inches='tight', transparent=True)
    # plt.savefig(f'results/plots/schematic/mds_univariate_{model_froi}_{fmri_froi}_whiteedge.png', dpi=300)
    # plt.savefig(f'results/plots/frontiers_poster/mds_univariate_{model_froi}_{fmri_froi}.pdf', dpi=300)
    plt.close()

Plotting MDS for median and median, metric: pearsonr
Plotting MDS for FFA and FFA, metric: pearsonr
Plotting MDS for PPA and PPA, metric: pearsonr
Plotting MDS for EBA and EBA, metric: pearsonr
Plotting MDS for median and median, metric: rdmr
Plotting MDS for FFA and FFA, metric: rdmr
Plotting MDS for PPA and PPA, metric: rdmr
Plotting MDS for EBA and EBA, metric: rdmr


In [23]:
# Plot
for metric in ['pearsonr', 'rdmr']:
    # Median
    model_froi = fmri_froi = 'median'
    print(f'Plotting MDS for {model_froi} and {fmri_froi}, metric: {metric}')
    mds_results = json.load(
        open(f'{results_dir}/uncorrected_mds_results_{metric}_{model_froi}_{fmri_froi}.json')
    )
    mds_coords = np.array(mds_results['mds_coordinates'])
    keys = mds_results['keys']

    human_color = 'royalblue'
    model_color = 'red'
    facecolors = [mcolors.to_rgba(human_color, alpha=0.75) if key in range(1, 9) else mcolors.to_rgba(model_color, alpha=0.75) for key in keys]
    plt.figure(figsize=(5, 5), dpi=300)
    plt.scatter(mds_coords[:, 0], mds_coords[:, 1], fc=facecolors, ec='none', s=100, linewidths=2)

    annotate_models = False  # Set to True to annotate models, False to not annotate
    if annotate_models:
        for i, key in enumerate(keys):
            if key in froi_models_data:
                plt.annotate(key, (mds_coords[i, 0], mds_coords[i, 1]), fontsize=8)

    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    plt.xticks([])
    plt.yticks([])
    # plt.xlabel('MDS Dimension 1')
    # plt.ylabel('MDS Dimension 2')
    # plt.legend(handles=[plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=human_color, markersize=10, label='Human'),
    #                     plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=model_color, markersize=10, label='Model')],
    #         loc='best')
    # plt.title(f'{model_froi} and {fmri_froi}')
    plt.axis('off')
    plt.gcf().patch.set_alpha(0.0)
    plt.tight_layout()
    # plt.show()
    plt.savefig(f'{plots_dir}/mds_uncorrected_{metric}_median_frois.pdf', dpi=300, bbox_inches='tight', transparent=True)
    plt.close()

# Individual frois
fig, axs = plt.subplots(2, len(model_fmri_frois), figsize=(5*len(model_fmri_frois), 5*2), dpi=300)
for midx, metric in enumerate(['pearsonr', 'rdmr']):
    for fidx, (model_froi, fmri_froi) in enumerate(model_fmri_frois):
        print(f'Plotting MDS for {model_froi} and {fmri_froi}, metric: {metric}')
        mds_results = json.load(
            open(f'{results_dir}/uncorrected_mds_results_{metric}_{model_froi}_{fmri_froi}.json')
        )
        mds_coords = np.array(mds_results['mds_coordinates'])
        keys = mds_results['keys']

        human_color = 'royalblue'
        model_color = 'red'
        facecolors = [mcolors.to_rgba(human_color, alpha=0.75) if key in range(1, 9) else mcolors.to_rgba(model_color, alpha=0.75) for key in keys]
        axs[midx, fidx].scatter(mds_coords[:, 0], mds_coords[:, 1], fc=facecolors, ec='none', s=100, linewidths=2)

        annotate_models = False  # Set to True to annotate models, False to not annotate
        if annotate_models:
            for i, key in enumerate(keys):
                if key in froi_models_data:
                    plt.annotate(key, (mds_coords[i, 0], mds_coords[i, 1]), fontsize=8)

        # axs[midx, fidx].spines['top'].set_visible(False)
        # axs[midx, fidx].spines['right'].set_visible(False)
        axs[midx, fidx].set_xticks([])
        axs[midx, fidx].set_yticks([])
        # plt.xlabel('MDS Dimension 1')
        # plt.ylabel('MDS Dimension 2')
        # plt.legend(handles=[plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=human_color, markersize=10, label='Human'),
        #                     plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=model_color, markersize=10, label='Model')],
        #         loc='best')
        # axs[midx, fidx].set_title(f'{model_froi} and {fmri_froi}')
        axs[midx, fidx].axis('off')
        # Write froi name in bottom left corner
        axs[midx, fidx].text(0.9, 0.1, model_froi, transform=axs[midx, fidx].transAxes, fontsize=12, verticalalignment='top')
plt.gcf().patch.set_alpha(0.0)
plt.tight_layout()
# plt.show()
plt.savefig(f'{plots_dir}/mds_uncorrected_pearsronr_rdmr_individual_frois.pdf', dpi=300, bbox_inches='tight', transparent=True)
plt.close()

Plotting MDS for median and median, metric: pearsonr
Plotting MDS for median and median, metric: rdmr
Plotting MDS for FFA and FFA, metric: pearsonr
Plotting MDS for PPA and PPA, metric: pearsonr
Plotting MDS for EBA and EBA, metric: pearsonr
Plotting MDS for FFA and FFA, metric: rdmr
Plotting MDS for PPA and PPA, metric: rdmr
Plotting MDS for EBA and EBA, metric: rdmr
